# EDA overview - remaining Vienna open-data POI files

Profiling every CSV in `data/raw/` except the Wiener Linien files (covered separately) to get a quick overview of the data. 
For details you can refer to the individual notebooks, which contain more detailed exploration and notes.

## Setup

In [1]:
import pandas as pd
import re
import glob
import os

pd.set_option("display.max_colwidth", 60)

RAW_DIR = "../data/raw"
all_files = sorted(
    f for f in glob.glob(os.path.join(RAW_DIR, "*.csv"))
    if "wienerlinien" not in os.path.basename(f)
)
[os.path.basename(f) for f in all_files]

['ADVENTMARKTOGD.csv',
 'BADESTELLENOGD.csv',
 'BUECHEREIOGD.csv',
 'DONAUINSPKTOGD.csv',
 'GRILLPLATZOGD.csv',
 'GRILLZONEOGD.csv',
 'HUNDEZONEOGD.csv',
 'MONUMENTALBRUOGD.csv',
 'MUSEUMOGD.csv',
 'PARKINFOOGD.csv',
 'SCHWIMMBADOGD.csv',
 'SPIELPLATZPUNKTOGD.csv',
 'SPORTSTAETTENOGD.csv',
 'WANDERWEGEOGD.csv',
 'WANDERWEGLAINZOGD.csv',
 'WANDERWEGLOBAUOGD.csv',
 'WIENTOURISMUSOGD.csv']

## Profiling function

Generic profiler that works across files without assuming a fixed schema:
- detects the geometry type from the `SHAPE` WKT string (POINT / LINESTRING / POLYGON / ...)
- extracts a rough bounding box directly via regex on all coordinate pairs (works
  for any WKT geometry type, not just points)
- flags known ArcGIS/SDE "junk" columns seen in `MUSEUMOGD.csv` (`FID`, `OBJECTID`,
  `SE_SDO_ROWID`, `SE_ANNO_CAD_DATA`) if present
- looks for likely name/address/category columns from a candidate list built from
  scanning all the headers
- reports null columns, fully-null columns, and duplicate rows

In [2]:
JUNK_COLS = {"FID", "OBJECTID", "SE_SDO_ROWID", "SE_ANNO_CAD_DATA"}
NAME_CANDIDATES = ["NAME", "BEZEICHNUNG", "BEZ_TEXT", "ANL_NAME", "BASIS_NAME", "KATEGORIE_TXT", "LAGE", "PARK"]
ADDRESS_CANDIDATES = ["ADRESSE", "STREET"]
CATEGORY_CANDIDATES = ["KATEGORIE", "TYP", "TYP_TXT", "CATEGORY_NAME", "SUBCATEGORY_NAME", "SPORTSTAETTEN_ART"]

def extract_geom_type(shape_series):
    for val in shape_series.dropna():
        m = re.match(r"^([A-Z]+)\s*\(", str(val).strip())
        if m:
            return m.group(1)
    return None

def extract_bbox(shape_series):
    coords = []
    for val in shape_series.dropna():
        coords.extend(re.findall(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(val)))
    if not coords:
        return None
    lons = [float(x) for x, y in coords]
    lats = [float(y) for x, y in coords]
    return (round(min(lons), 3), round(max(lons), 3), round(min(lats), 3), round(max(lats), 3))

def profile_csv(path):
    df = pd.read_csv(path)
    cols = list(df.columns)
    n_rows, n_cols = df.shape

    junk_present = sorted(JUNK_COLS & set(cols))
    name_col = next((c for c in NAME_CANDIDATES if c in cols), None)
    addr_col = next((c for c in ADDRESS_CANDIDATES if c in cols), None)
    cat_cols = [c for c in CATEGORY_CANDIDATES if c in cols]
    has_bezirk = "BEZIRK" in cols
    has_shape = "SHAPE" in cols

    geom_type, bbox = None, None
    if has_shape:
        geom_type = extract_geom_type(df["SHAPE"])
        bbox = extract_bbox(df["SHAPE"])

    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    fully_null_cols = list(null_counts[null_counts == n_rows].index)
    dup_rows = int(df.duplicated().sum())
    dup_name = int(df[name_col].duplicated().sum()) if name_col else None

    return {
        "file": os.path.basename(path),
        "n_rows": n_rows,
        "n_cols": n_cols,
        "geom_type": geom_type,
        "bbox_lon_lat": bbox,
        "has_BEZIRK": has_bezirk,
        "name_col": name_col,
        "address_col": addr_col,
        "category_cols": ", ".join(cat_cols) if cat_cols else None,
        "junk_cols_present": ", ".join(junk_present) if junk_present else None,
        "fully_null_cols": ", ".join(fully_null_cols) if fully_null_cols else None,
        "cols_with_some_nulls": ", ".join(cols_with_nulls.index) if len(cols_with_nulls) else None,
        "dup_rows": dup_rows,
        "dup_on_name_col": dup_name,
    }

## Run the profiler across all files

In [3]:
profiles = [profile_csv(f) for f in all_files]
summary_df = pd.DataFrame(profiles)
summary_df

,file,n_rows,n_cols,geom_type,bbox_lon_lat,has_BEZIRK,name_col,address_col,category_cols,junk_cols_present,fully_null_cols,cols_with_some_nulls,dup_rows,dup_on_name_col
0,ADVENTMARKTOGD.csv,18,10,POINT,"(16.313, 16.405, 48.177, 48.257)",False,BEZEICHNUNG,ADRESSE,NaN,"FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,"WEBLINK1, SE_ANNO_CAD_DATA",0,0
1,BADESTELLENOGD.csv,32,14,POINT,"(16.349, 16.541, 48.167, 48.284)",True,BEZEICHNUNG,NaN,TYP,"FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID",SE_ANNO_CAD_DATA,"BEZIRK, SICHTTIEFE, SE_ANNO_CAD_DATA",0,0
2,BUECHEREIOGD.csv,37,16,POINT,"(16.271, 16.509, 48.136, 48.278)",True,NAME,ADRESSE,NaN,"FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID",SE_ANNO_CAD_DATA,"OEFFNUNGSZEITEN5, OEFFNUNGSZEITEN6, EMAIL, SE_ANNO_CAD_DATA",0,0
3,DONAUINSPKTOGD.csv,369,8,POINT,"(16.348, 16.51, 48.164, 48.297)",False,BEZEICHNUNG,NaN,"TYP, TYP_TXT","FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,"ORTSBESCHREIBUNG, SE_ANNO_CAD_DATA",0,290
4,GRILLPLATZOGD.csv,17,8,POINT,"(16.226, 16.482, 48.15, 48.279)",False,LAGE,NaN,NaN,"FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,SE_ANNO_CAD_DATA,0,0
5,GRILLZONEOGD.csv,3,7,POLYGON,"(16.4, 16.464, 48.195, 48.239)",False,LAGE,NaN,NaN,"FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,SE_ANNO_CAD_DATA,0,1
6,HUNDEZONEOGD.csv,439,11,POINT,"(16.22, 16.519, 48.122, 48.294)",False,PARK,NaN,TYP,"FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,"TELEFON, EINFRIEDUNG, WEBLINK1, SE_ANNO_CAD_DATA",0,92
7,MONUMENTALBRUOGD.csv,58,15,POINT,"(16.27, 16.399, 48.152, 48.278)",False,BASIS_NAME,NaN,NaN,"FID, OBJECTID, SE_ANNO_CAD_DATA","VERWALTUNG, DENKMAL, BETRIEB_VON, BETRIEB_BIS, SE_ANNO_C...","BASIS_NAME, VERWALTUNG, BAUJAHR, DENKMAL, KUENSTLER, BET...",0,2
8,MUSEUMOGD.csv,137,8,POINT,"(16.256, 16.508, 48.145, 48.264)",True,NAME,ADRESSE,NaN,"FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID",SE_ANNO_CAD_DATA,"WEITERE_INF, SE_ANNO_CAD_DATA",0,1
9,PARKINFOOGD.csv,1051,13,POINT,"(16.212, 16.535, 48.126, 48.303)",True,ANL_NAME,NaN,NaN,"FID, OBJECTID, SE_ANNO_CAD_DATA",SE_ANNO_CAD_DATA,"OEFF_ZEITEN, SE_ANNO_CAD_DATA",0,2


## Common ground - computed directly from the table above

These cells answer: how much do these files actually share?

In [4]:
print("Geometry type distribution:")
print(summary_df["geom_type"].value_counts(dropna=False))

print("\nFiles WITHOUT a BEZIRK (district) column:")
print(summary_df.loc[~summary_df["has_BEZIRK"], "file"].tolist())

print("\nFiles WITH a BEZIRK (district) column:")
print(summary_df.loc[summary_df["has_BEZIRK"], "file"].tolist())

Geometry type distribution:
geom_type
POINT         13
LINESTRING     3
POLYGON        1
Name: count, dtype: int64

Files WITHOUT a BEZIRK (district) column:
['ADVENTMARKTOGD.csv', 'DONAUINSPKTOGD.csv', 'GRILLPLATZOGD.csv', 'GRILLZONEOGD.csv', 'HUNDEZONEOGD.csv', 'MONUMENTALBRUOGD.csv', 'SPORTSTAETTENOGD.csv', 'WANDERWEGEOGD.csv', 'WANDERWEGLAINZOGD.csv', 'WANDERWEGLOBAUOGD.csv', 'WIENTOURISMUSOGD.csv']

Files WITH a BEZIRK (district) column:
['BADESTELLENOGD.csv', 'BUECHEREIOGD.csv', 'MUSEUMOGD.csv', 'PARKINFOOGD.csv', 'SCHWIMMBADOGD.csv', 'SPIELPLATZPUNKTOGD.csv']


In [ ]:
print("SE_ANNO_CAD_DATA present (usually 100% null, ArcGIS export artifact):")
has_anno = summary_df["junk_cols_present"].fillna("").str.contains("SE_ANNO_CAD_DATA")
print(summary_df.loc[has_anno, "file"].tolist())

print("\nSE_SDO_ROWID vs OBJECTID - which numeric-ID convention each file uses:")
print(summary_df["junk_cols_present"])

SE_ANNO_CAD_DATA present (usually 100% null, ArcGIS export artifact):
['ADVENTMARKTOGD.csv', 'BADESTELLENOGD.csv', 'BUECHEREIOGD.csv', 'DONAUINSPKTOGD.csv', 'GRILLPLATZOGD.csv', 'GRILLZONEOGD.csv', 'HUNDEZONEOGD.csv', 'MONUMENTALBRUOGD.csv', 'MUSEUMOGD.csv', 'PARKINFOOGD.csv', 'SCHWIMMBADOGD.csv', 'SPIELPLATZPUNKTOGD.csv', 'SPORTSTAETTENOGD.csv', 'WANDERWEGEOGD.csv', 'WANDERWEGLAINZOGD.csv', 'WANDERWEGLOBAUOGD.csv', 'WIENTOURISMUSOGD.csv']

SE_SDO_ROWID vs OBJECTID — which numeric-ID convention each file uses:
0         FID, OBJECTID, SE_ANNO_CAD_DATA
1     FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID
2     FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID
3         FID, OBJECTID, SE_ANNO_CAD_DATA
4         FID, OBJECTID, SE_ANNO_CAD_DATA
5         FID, OBJECTID, SE_ANNO_CAD_DATA
6         FID, OBJECTID, SE_ANNO_CAD_DATA
7         FID, OBJECTID, SE_ANNO_CAD_DATA
8     FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID
9         FID, OBJECTID, SE_ANNO_CAD_DATA
10    FID, SE_ANNO_CAD_DATA, SE_SDO_ROWID
11        FID, OBJECTI

In [6]:
print("Files with no detected name-like column (name_col is None):")
print(summary_df.loc[summary_df["name_col"].isnull(), "file"].tolist())

print("\nFiles with no detected address-like column:")
print(summary_df.loc[summary_df["address_col"].isnull(), "file"].tolist())

print("\nFiles with duplicate rows:")
print(summary_df.loc[summary_df["dup_rows"] > 0, ["file", "dup_rows"]])

Files with no detected name-like column (name_col is None):
[]

Files with no detected address-like column:
['BADESTELLENOGD.csv', 'DONAUINSPKTOGD.csv', 'GRILLPLATZOGD.csv', 'GRILLZONEOGD.csv', 'HUNDEZONEOGD.csv', 'MONUMENTALBRUOGD.csv', 'PARKINFOOGD.csv', 'SPIELPLATZPUNKTOGD.csv', 'WANDERWEGEOGD.csv', 'WANDERWEGLAINZOGD.csv', 'WANDERWEGLOBAUOGD.csv']

Files with duplicate rows:
Empty DataFrame
Columns: [file, dup_rows]
Index: []


## Per-file detail

Same style of look as the initial `MUSEUMOGD` notebook - columns, sample rows, and
null counts for every remaining file. `WIENTOURISMUSOGD.csv` is handled separately
further down since it doesn't fit the common pattern (must be some different data gathering 
process by the city).

In [ ]:
for f in all_files:
    name = os.path.basename(f)
    if name == "WIENTOURISMUSOGD.csv":
        continue  # separate deep-dive below, different schema entirely
    df = pd.read_csv(f)
    print("=" * 90)
    print(name, "-", df.shape[0], "rows,", df.shape[1], "columns")
    print("=" * 90)
    print("columns:", list(df.columns))
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print("columns with nulls:", dict(nulls) if len(nulls) else "none")
    print(df.head(2).to_string())
    print()

ADVENTMARKTOGD.csv — 18 rows, 10 columns
columns: ['FID', 'OBJECTID', 'SHAPE', 'BEZEICHNUNG', 'ADRESSE', 'DATUM', 'OEFFNUNGSZEIT', 'WEBLINK1', 'SE_ANNO_CAD_DATA', 'SILVESTERMARKT']
columns with nulls: {'WEBLINK1': np.int64(7), 'SE_ANNO_CAD_DATA': np.int64(18)}
                    FID  OBJECTID                                          SHAPE                       BEZEICHNUNG           ADRESSE                  DATUM                                                     OEFFNUNGSZEIT                          WEBLINK1  SE_ANNO_CAD_DATA  SILVESTERMARKT
0  ADVENTMARKTOGD.79224     79224   POINT (16.359357012101153 48.21057098643211)  Christkindlmarkt am Rathausplatz  1., Rathausplatz  14.11. bis 24.12.2025  14.11. bis 26.12. von 10 bis 22 Uhr, 24.12. von 10 bis 18.30 Uhr  https://www.christkindlmarkt.at/               NaN               0
1  ADVENTMARKTOGD.79225     79225  POINT (16.365406954481536 48.211770469125106)    38. Altwiener Christkindlmarkt       1., Freyung      14.11. bis 23.12.    

## Deep dive - `WIENTOURISMUSOGD.csv`

Flagged upfront as the odd one out: different schema (separate `GEOLAT`/`GEOLONG`
columns alongside `SHAPE`, no `BEZIRK`, but `POSTALCODE`/`CATEGORY_NAME`/
`SUBCATEGORY_NAME`), and by far the largest file (~750KB vs <350KB for everything
else). Likely a broad tourism-infrastructure collection (hotels, sights, etc.)
rather than one POI type, so it needs its own category breakdown before deciding
if/how it fits the KG's initial ~5-category scope.

In [8]:
tourism = pd.read_csv("../data/raw/WIENTOURISMUSOGD.csv")
print("shape:", tourism.shape)
print("columns:", list(tourism.columns))
tourism.head(3)

shape: (2860, 15)
columns: ['FID', 'OBJECTID', 'NAME', 'STREET', 'POSTALCODE', 'EMAIL', 'PHONE', 'WEBSITE', 'CATEGORY_NAME', 'SUBCATEGORY_NAME', 'UID_', 'GEOLAT', 'GEOLONG', 'SHAPE', 'SE_ANNO_CAD_DATA']


,FID,OBJECTID,NAME,STREET,POSTALCODE,EMAIL,PHONE,WEBSITE,CATEGORY_NAME,SUBCATEGORY_NAME,UID_,GEOLAT,GEOLONG,SHAPE,SE_ANNO_CAD_DATA
0,WIENTOURISMUSOGD.719830,719830,The Stellas,Landstraßer Hauptstraße 44,1030,NaN,+43 1 710 67 73,https://www.stellas.at,"Essen, Trinken & Nightlife",Restaurant,f38c9d6497411462e44d293d9d4195cb,48.202854,16.389578,POINT (16.389577599908975 48.20285356589163),NaN
1,WIENTOURISMUSOGD.719831,719831,Café Friedlich,Untere Weißgerberstraße 13,1030,NaN,NaN,https://www.friedlich.wien,"Essen, Trinken & Nightlife",Haubenlokal,03a133d96b8f31de3fd24ddce01d81ee,48.211134,16.393276,POINT (16.393275670411143 48.211133916238566),NaN
2,WIENTOURISMUSOGD.719832,719832,Karmeliterviertel,Karmeliterplatz,1020,NaN,NaN,NaN,Sightseeing,Sehenswürdigkeit,8cbdaff81ec65b8bbd8b84d1d229eac8,48.215702,16.380030,POINT (16.380030371931998 48.215702432816926),NaN


In [9]:
print("Top-level CATEGORY_NAME value counts:")
print(tourism["CATEGORY_NAME"].value_counts())

Top-level CATEGORY_NAME value counts:
CATEGORY_NAME
Essen, Trinken & Nightlife        1036
Sightseeing                        749
Unterkünfte                        399
Infrastruktur                      209
Freizeit, Unterhaltung & Sport     174
Musik & Theater                    156
Verkehr & Transport                 75
Touren & Guides                     62
Name: count, dtype: int64


In [10]:
print("SUBCATEGORY_NAME value counts (top 30):")
print(tourism["SUBCATEGORY_NAME"].value_counts().head(30))
print("\n...total distinct subcategories:", tourism["SUBCATEGORY_NAME"].nunique())

SUBCATEGORY_NAME value counts (top 30):
SUBCATEGORY_NAME
Restaurant                335
Hotel                     305
Sehenswürdigkeit          207
Kaffeehaus                163
Kirche & Kapelle          152
Haubenlokal               138
Museum                    137
Botschaft                 111
Bar                        92
Gasthaus & Beisl           92
Galerie & Kunsthandel      84
Heuriger                   73
Park & Garten              63
Touren & Guides            62
Pension                    54
Sportanlage                47
Eissalon                   43
Schloss & Palais           43
Theater                    42
Konzertsaal                39
Appartement                38
Sonstiges                  36
Club & Disco               36
Bildungseinrichtung        34
Imbiss & Fast Food         33
Kleinbühne                 33
Gedenkstätte & Denkmal     32
Eventlocation              30
Schwimmbad                 29
Busparkplatz               23
Name: count, dtype: int64

...total distinc

In [11]:
# Sanity-check: do GEOLAT/GEOLONG agree with the coordinates embedded in SHAPE?
sample = tourism.dropna(subset=["SHAPE", "GEOLAT", "GEOLONG"]).head(5)
for _, row in sample.iterrows():
    m = re.match(r"POINT \(([\-0-9.]+) ([\-0-9.]+)\)", str(row["SHAPE"]))
    shape_lon, shape_lat = (float(m.group(1)), float(m.group(2))) if m else (None, None)
    print(f"GEOLAT={row['GEOLAT']} GEOLONG={row['GEOLONG']}  |  SHAPE lon={shape_lon} lat={shape_lat}")

GEOLAT=48.2028535 GEOLONG=16.38957759  |  SHAPE lon=16.389577599908975 lat=48.20285356589163
GEOLAT=48.21113385 GEOLONG=16.39327566  |  SHAPE lon=16.393275670411143 lat=48.211133916238566
GEOLAT=48.21570237 GEOLONG=16.38003037  |  SHAPE lon=16.380030371931998 lat=48.215702432816926
GEOLAT=48.2088475 GEOLONG=16.371284  |  SHAPE lon=16.37128400495679 lat=48.208847565712176
GEOLAT=48.2041786 GEOLONG=16.3576164  |  SHAPE lon=16.357616405246027 lat=48.2041786662107


## Findings

**Row counts vary ** - from `GRILLZONEOGD.csv` at just 3 rows (only 3
formal BBQ zones citywide) up to `WIENTOURISMUSOGD.csv` at 2,860 and
`SPORTSTAETTENOGD.csv` at 1,543. `GRILLZONEOGD` is probably too small to stand as
its own KG category and makes more sense merged with `GRILLPLATZOGD.csv` (17 rows,
individual BBQ spots) into one "Grillen" category.

**Junk columns are universal but split into two conventions.** `SE_ANNO_CAD_DATA`
is present and 100% null in all 17 files - safe to drop everywhere, no exceptions.
The numeric-ID column differs: 13 files use `OBJECTID`, while exactly 4
(`BADESTELLENOGD`, `BUECHEREIOGD`, `MUSEUMOGD`, `SCHWIMMBADOGD`) use `SE_SDO_ROWID`
instead. These same 4 files are also the only ones with both `BEZIRK`
and a structured `ADRESSE` - suggesting they were exported from a slightly
different pipeline/template than the other 13, despite all coming from
the same Vienna Open Data portal.

**Geometry types: 13 POINT, 3 LINESTRING, 1 POLYGON.** The three
`WANDERWEG*OGD.csv` files are lines, `GRILLZONEOGD.csv` is the one polygon set, 
everything else is points. Hence these will need different proximity logic in the
to be build reasomning layer.

**`BEZIRK` (district) and structured addresses are the exception, not the norm :(** -
only 6 of 17 files have `BEZIRK` (`BADESTELLEN`, `BUECHEREI`, `MUSEUM`, `PARKINFO`,
`SCHWIMMBAD`, `SPIELPLATZPUNKT`) and the same 6 (roughly) have an address field.
The rest rely on coordinates alone or a free-text location description
(`LAGE`/`ORTSBESCHREIBUNG`/`PARK`). Not a problem for spatial reasoning (only
needs coordinates), but means district/address can't be directly used for filtering
if not differently derived!

**The "name" field is inconsistent, and sometimes isn't really a name.** Every
file now has *some* field detected as name-like, but for `SPORTSTAETTENOGD.csv`
the best available field is `KATEGORIE_TXT` - a sport-facility category (e.g.
"Fußballplatz"), not a unique name per location, which is why 1,538 of 1,543 rows
look like "duplicates" on that column. Same caveat applies more mildly to
`HUNDEZONEOGD.csv` (`PARK`, 92 repeats - multiple dog zones share a park name)...

**`WIENTOURISMUSOGD.csv` confirmed as the extensive outlier**, 
Different schema (`GEOLAT`/`GEOLONG` alongside `SHAPE`, no `BEZIRK`), by far the
largest file, and wide-reaching: 8 top-level `CATEGORY_NAME` values (dominated by
"Essen, Trinken & Nightlife" at 1,036 and "Sightseeing" at 749) and 56 distinct
`SUBCATEGORY_NAME` values.